## here we will use muliple features for prediction instead of just one like in version one where only description was used

In [40]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [41]:
df = pd.read_excel("../data/sample_transactions.xlsx")
df.head()

,transaction_id,date,day_of_week,month,description,merchant,amount,category,payment_mode,transaction_channel,account_type,city,is_weekend,is_anomaly
0,TXN00001,2024-01-02,Tuesday,January,POS TXN 9289,Cafe Coffee Day,8123.00,Food,Cash,ATM,Savings,Delhi,False,True
1,TXN00002,2024-01-02,Tuesday,January,dunzo delivery,Dunzo,315.93,Food,NEFT,Web,Savings,Delhi,False,False
2,TXN00003,2024-01-02,Tuesday,January,property tax payment,BBMP,4619.46,Utilities,Debit Card,Auto Debit,Savings,Delhi,False,False
3,TXN00004,2024-01-02,Tuesday,January,monthly rent,Housing Society,14768.18,Rent,Net Banking,Auto Debit,Savings,Delhi,False,False
4,TXN00005,2024-01-02,Tuesday,January,monthly rent,Housing Society,20437.67,Rent,Auto Debit,Auto Debit,Savings,Delhi,False,False


In [42]:
## checking correlation of numerical features with category

print(df.groupby('category')['amount'].mean().sort_values())

category
Food               927.652878
Entertainment     1499.183960
Fuel              1985.153934
Bills             2467.164545
Groceries         2733.426923
Personal Care     2853.718000
Travel            2912.124257
Utilities         3533.739595
Healthcare        4227.100656
Fitness           4735.143750
Shopping          5871.068387
Insurance         8204.649118
Rent             15474.642533
Investments      15573.175082
Education        17838.963617
Name: amount, dtype: float64


In [43]:
print(df.groupby('category')['is_weekend'].mean().sort_values())

category
Personal Care    0.175000
Education        0.191489
Fitness          0.225000
Rent             0.240000
Utilities        0.256757
Groceries        0.274725
Investments      0.278689
Food             0.284173
Insurance        0.294118
Healthcare       0.311475
Travel           0.326733
Shopping         0.335484
Bills            0.340909
Fuel             0.377049
Entertainment    0.485149
Name: is_weekend, dtype: float64


In [44]:
## for payement mode
print(df.groupby('category')['payment_mode'].value_counts(normalize=True).unstack())

payment_mode   Auto Debit      Cash  Credit Card  Debit Card      IMPS  \
category                                                                 
Bills            0.102273  0.090909     0.090909    0.125000  0.170455   
Education        0.085106  0.127660     0.063830    0.255319  0.085106   
Entertainment    0.148515  0.158416     0.118812    0.128713  0.138614   
Fitness          0.250000  0.125000     0.125000    0.175000  0.125000   
Food             0.100719  0.154676     0.093525    0.118705  0.179856   
Fuel                  NaN  0.344262     0.262295    0.131148       NaN   
Groceries        0.131868  0.120879     0.120879    0.098901  0.142857   
Healthcare       0.196721  0.131148     0.114754    0.147541  0.081967   
Insurance        0.205882       NaN          NaN         NaN  0.323529   
Investments      0.213115       NaN          NaN         NaN  0.360656   
Personal Care    0.125000  0.075000     0.150000    0.025000  0.075000   
Rent             0.226667       NaN   

In [45]:
## days of week
print(df.groupby('category')['day_of_week'].value_counts(normalize=True).unstack())


day_of_week      Friday    Monday  Saturday    Sunday  Thursday   Tuesday  \
category                                                                    
Bills          0.125000  0.136364  0.170455  0.170455  0.193182  0.113636   
Education      0.127660  0.212766  0.148936  0.042553  0.170213  0.170213   
Entertainment  0.138614  0.079208  0.277228  0.207921  0.108911  0.099010   
Fitness        0.200000  0.100000  0.150000  0.075000  0.100000  0.175000   
Food           0.143885  0.143885  0.136691  0.147482  0.143885  0.143885   
Fuel           0.114754  0.180328  0.213115  0.163934  0.131148  0.081967   
Groceries      0.192308  0.082418  0.142857  0.131868  0.115385  0.159341   
Healthcare     0.131148  0.098361  0.131148  0.180328  0.114754  0.131148   
Insurance      0.088235  0.117647  0.176471  0.117647  0.176471  0.147059   
Investments    0.114754  0.131148  0.114754  0.163934  0.114754  0.180328   
Personal Care  0.175000  0.200000  0.150000  0.025000  0.200000  0.125000   

### conclusions

amount, description and payment mode are correlated to the category 

whereas other features like day of week, month, city, merchant do not matter that much so they wont be used for prediction

description will be done using TF-IDF
amount -> numerical so can be done using a regressor
payment mode -> categorical but requres encoding

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [47]:
## preparing the data

#chanding date from string to date format
df['date'] = pd.to_datetime(df['date'])
print(df['date'].dtype)

datetime64[us]


In [48]:
## cleaning the description
df['description'] = df['description'].str.lower().str.strip()
print(df['description'].head())

0            pos txn 9289
1          dunzo delivery
2    property tax payment
3            monthly rent
4            monthly rent
Name: description, dtype: str


In [49]:
df.head()

,transaction_id,date,day_of_week,month,description,merchant,amount,category,payment_mode,transaction_channel,account_type,city,is_weekend,is_anomaly
0,TXN00001,2024-01-02,Tuesday,January,pos txn 9289,Cafe Coffee Day,8123.00,Food,Cash,ATM,Savings,Delhi,False,True
1,TXN00002,2024-01-02,Tuesday,January,dunzo delivery,Dunzo,315.93,Food,NEFT,Web,Savings,Delhi,False,False
2,TXN00003,2024-01-02,Tuesday,January,property tax payment,BBMP,4619.46,Utilities,Debit Card,Auto Debit,Savings,Delhi,False,False
3,TXN00004,2024-01-02,Tuesday,January,monthly rent,Housing Society,14768.18,Rent,Net Banking,Auto Debit,Savings,Delhi,False,False
4,TXN00005,2024-01-02,Tuesday,January,monthly rent,Housing Society,20437.67,Rent,Auto Debit,Auto Debit,Savings,Delhi,False,False


In [50]:
## defining input and output features

X = df[['description', 'amount', 'payment_mode']]
y = df['category']

print(X.head(),"\n")
print(y.head())

            description    amount payment_mode
0          pos txn 9289   8123.00         Cash
1        dunzo delivery    315.93         NEFT
2  property tax payment   4619.46   Debit Card
3          monthly rent  14768.18  Net Banking
4          monthly rent  20437.67   Auto Debit 

0         Food
1         Food
2    Utilities
3         Rent
4         Rent
Name: category, dtype: str


In [51]:
## training testing split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

print("training dataset shape: ",X_train.shape)
print("testing dataset shape: ", X_test.shape)

training dataset shape:  (1118, 3)
testing dataset shape:  (280, 3)


In [ ]:
## making the column transformer



preprocessor = ColumnTransformer(transformers=[
    ('text', TfidVectorizer(ngram_range=(1,2)), 'description')
])

NameError: name 'TfidVectorizer' is not defined